# Week 2 Day 2 — LangChain: Tools, Chains, Memory & Framework Agent

Rebuilds Day 1's raw-Python Gemini agent using LangChain, then adds chaining,
real memory, multiple tools (including a local data source), a full reasoning
trace, and structured output.

Uses `langchain-google-genai` so it plugs into the same free `GEMINI_API_KEY`
from Day 1's `.env` file — copy that `.env` file into this folder first.

In [2]:
# Setup — reuses Day 1's .env file (must be copied into this folder)
import os
import json

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool, ToolException
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in .env — reuse the same key as Day 1.")

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
print("LLM ready:", llm.model)

LLM ready: gemini-3.6-flash


## Task 1: LangChain Setup & Core Concepts

| Day 1 (raw Python) | LangChain equivalent |
|---|---|
| `call_gemini_with_tools()` | `ChatGoogleGenerativeAI` (LLM wrapper) |
| `TOOLS` list + `execute_tool()` if/elif | `@tool` decorator |
| `for step in range(max_iterations)` loop | `create_agent(...)` |
| `scratchpad` dict | `checkpointer=InMemorySaver()` + `thread_id` |

### LCEL pipe (`|`) demo

In [3]:
# A basic prompt -> LLM -> parser pipeline using LCEL
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("user", "{question}")
])
lcel_chain = prompt | llm | StrOutputParser()
print(lcel_chain.invoke({"question": "In one sentence, what is LangChain?"}))

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


LangChain is an open-source software framework designed to simplify the creation of applications powered by large language models (LLMs) by connecting them to external data sources, APIs, and computational tools.


**What `|` does under the hood:** every component is a `Runnable` sharing an
`.invoke()` interface. `|` builds a `RunnableSequence` that feeds each
component's output into the next — sugar for
`StrOutputParser().invoke(llm.invoke(prompt.invoke(input)))`.

## Task 2: Define & Register Tools

Three tools: `calculator` and `get_weather` are reused from Day 1. The third,
`get_product_price`, is new — it reads from a local JSON "database"
(`products.json`), simulating a real external data source.

In [4]:
# Create a small local JSON "database" of products
PRODUCTS_DB_PATH = "products.json"
products_data = {
    "laptop a": {"price_usd": 650, "specs": "8GB RAM, 256GB SSD, 14-inch"},
    "laptop b": {"price_usd": 950, "specs": "16GB RAM, 512GB SSD, 15-inch"},
    "laptop c": {"price_usd": 1200, "specs": "32GB RAM, 1TB SSD, 16-inch"},
}
with open(PRODUCTS_DB_PATH, "w") as f:
    json.dump(products_data, f, indent=2)
print(f"Wrote {PRODUCTS_DB_PATH} with {len(products_data)} products")

# Reused from Day 1
FAKE_WEATHER_DB = {
    "lahore": {"temp_c": 38, "condition": "Sunny"},
    "karachi": {"temp_c": 33, "condition": "Humid"},
    "islamabad": {"temp_c": 29, "condition": "Cloudy"},
    "london": {"temp_c": 18, "condition": "Rainy"},
    "tokyo": {"temp_c": 27, "condition": "Clear"},
    "new york": {"temp_c": 22, "condition": "Sunny"},
    "sydney": {"temp_c": 18, "condition": "Rainy"},
}

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression (+, -, *, /, parentheses).
    Use this whenever the user asks for a numeric calculation.
    Input must be a plain math expression string, e.g. '15 * 7 - 2'.
    """
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression) <= allowed_chars:
        return f"ERROR: invalid characters in expression '{expression}'"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"ERROR: could not evaluate expression - {e}"

@tool
def get_weather(city: str) -> str:
    """Get the current weather (temperature in Celsius and condition) for a given city.
    Use this whenever the user asks about weather in a specific city.
    Input is the city name, e.g. 'Lahore' or 'Tokyo'.
    """
    key = city.strip().lower()
    if key not in FAKE_WEATHER_DB:
        return f"ERROR: no weather data found for city '{city}'"
    data = FAKE_WEATHER_DB[key]
    return f"{data['temp_c']}C, {data['condition']}"

@tool
def get_product_price(product_name: str) -> str:
    """Look up the price and specs of a product from the local product database.
    Use this whenever the user asks about the price or specs of a named product
    (e.g. 'Laptop A', 'Laptop B'). Input is the product name.
    Raises an error if the product name is not found, so Task 5's error
    handling has something real to catch.
    """
    with open(PRODUCTS_DB_PATH) as f:
        db = json.load(f)
    key = product_name.strip().lower()
    if key not in db:
        raise ToolException(f"No product named '{product_name}' in the database.")
    info = db[key]
    return f"{product_name}: ${info['price_usd']} ({info['specs']})"

TOOLS = [calculator, get_weather, get_product_price]
for t in TOOLS:
    print(f"- {t.name}: {t.description[:70]}...")

Wrote products.json with 3 products
- calculator: Evaluate a basic arithmetic expression (+, -, *, /, parentheses).
    ...
- get_weather: Get the current weather (temperature in Celsius and condition) for a g...
- get_product_price: Look up the price and specs of a product from the local product databa...


**Why the docstring matters:** LangChain sends each tool's `name`,
`description`, and argument schema to the model as part of the
function-calling payload — the docstring above is literally the text the
model reads to decide when to call the tool and what to pass in.

## Task 3: Build an Agent with `create_agent`

In [5]:
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a helpful assistant with access to tools. "
                  "Use tools when needed, and give a clear final answer.",
)

trace_result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "Look up the weather in Lahore and Tokyo and tell me which is warmer."}]}
)

# No verbose= flag in the new API — the trace is every message in result["messages"]
for m in trace_result["messages"]:
    role = m.__class__.__name__
    calls = getattr(m, "tool_calls", None)
    if calls:
        print(f"[{role}] requests tool call(s): {calls}")
    else:
        print(f"[{role}] {m.content}")

print("\nFINAL ANSWER:", trace_result["messages"][-1].content)

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[HumanMessage] Look up the weather in Lahore and Tokyo and tell me which is warmer.
[AIMessage] requests tool call(s): [{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': 'call_896356', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_896357', 'type': 'tool_call'}]
[ToolMessage] 38C, Sunny
[ToolMessage] 27C, Clear
[AIMessage] [{'type': 'text', 'text': 'The weather in Lahore is 38°C (Sunny) and in Tokyo it is 27°C (Clear). \n\n**Lahore** is warmer than Tokyo by 11°C.', 'extras': {'signature': 'EocCCoQCARFNMg9pBk+VmLCHh946KNgt/1KKWPXXZ5yZ3ht51QRJjbw1IP2o/C+flexzrn6d1Mts7cHEADJaU7nShOYtuQvjkfBnxYfodvLBaUKttHbamZbxh1tUXvBwPJogeY9m+A4HVd+QTbJVnoBTm3bcIii+TllOXhCI2Df9hslWPGoUmSBevyDhB2O+5aX7Ex5T43ORLSREgQDDZVn4lwBVUIK357MOoimaj6Ema8SsVDBhisUHciZ8Aa+iCTvoKtDXL40jhKA+jQOLIxci9P/NfSIBgbgtLr/6l/IuTApHVWPwCV8h//J9Rte+8HwhJhLu8s68kQKhzJkqpYwlLBY7uuYG/XI='}}]

FINAL ANSWER: [{'type': 'text', 'text': 'The weather in Lahore is 38°C (Sunny) and in Tokyo it is

### Annotated trace

| Trace line | ReAct stage |
|---|---|
| Model calls `get_weather(city="Lahore")` | Reason -> Act |
| Tool returns `38C, Sunny` | Observe |
| Model calls `get_weather(city="Tokyo")` | Reason -> Act |
| Tool returns `27C, Clear` | Observe |
| Model compares 38C vs 27C, gives final answer | Reason -> Final answer |

**Similar to Day 1:** the loop is still reason -> act -> observe -> repeat.
**Hidden now:** the exact prompt text sent to Gemini and the parsing logic —
`create_agent` builds and hides both inside a compiled LangGraph.

## Task 4: Add Memory

In [6]:
agent_with_memory = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a helpful assistant with access to tools. "
                  "Use tools when needed, and give a clear final answer.",
    checkpointer=InMemorySaver(),
)

thread = {"configurable": {"thread_id": "budget-laptop-chat"}}

turn1 = agent_with_memory.invoke({"messages": [{"role": "user", "content": "Find the price of Laptop A"}]}, thread)
print("Turn 1:", turn1["messages"][-1].content, "\n")

turn2 = agent_with_memory.invoke({"messages": [{"role": "user", "content": "Now compare it to Laptop B"}]}, thread)
print("Turn 2:", turn2["messages"][-1].content, "\n")

turn3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Which one should I recommend to a budget-conscious client?"}]},
    thread,
)
print("Turn 3:", turn3["messages"][-1].content)

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 1: [{'type': 'text', 'text': 'The price of Laptop A is **$650**. \n\n**Specs:**\n* **RAM:** 8GB\n* **Storage:** 256GB SSD\n* **Screen:** 14-inch', 'extras': {'signature': 'EukBCuYBARFNMg+Jsg9D9PK05V8ghVH/eUu1+KxxseNGMROfoJRGe5+c57lQiQBahK61Io0zkIqMfLopCnJSZ6ksRYS8MqXuZUkwNAQ7wjRn6SmVdclWfnlxvBluPxKtyCwPxOQXq+OZzLvQpTQzguKNM1NUo/5Q6zF9eyRaXXhli6ZtsZoqk/lt75ihbsbmk0uuae/JfzsbZvKt9M877OOoAm5zIzauBq2SCiH+BsQEHluZOkrZHHM1+Ft+lYwon9UHDxLHltPnytBf29bOH/AxVzenUJ2Jpe82EAYcEwL0YU8fFatmTd1TOKE='}}] 



d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 2: [{'type': 'text', 'text': 'Here is a comparison between **Laptop A** and **Laptop B**:\n\n| Feature | Laptop A | Laptop B |\n| :--- | :--- | :--- |\n| **Price** | $650 | $950 |\n| **RAM** | 8 GB | 16 GB |\n| **Storage** | 256 GB SSD | 512 GB SSD |\n| **Display Size** | 14-inch | 15-inch |\n\n### Key Differences:\n* **Price:** Laptop B costs **$300 more** than Laptop A.\n* **Performance & Memory:** Laptop B has double the RAM (16 GB vs 8 GB), making it better for multitasking and heavier workloads.\n* **Storage:** Laptop B offers double the storage capacity (512 GB SSD vs 256 GB SSD).\n* **Display:** Laptop B has a slightly larger screen (15-inch vs 14-inch).', 'extras': {'signature': 'Ep0DCpoDARFNMg+nVZ7U71WhOZBzT79QBT0vOKonmnTxJ0dsxE9KJyIpcPLgr05jss0DjyJGvUmkDDd3+zIsIaoum3SZAIfVzUQ2x1SxrOvzac9uL7ZelAmB+LH+sWvXVJ/ggjP982LnMw3xYqxGWGZgxbWj5FE5ZuNze4BgrfqiB66Vnlwh61TSABpUdU6qu3mayPYgLCc2OtwSc1Ag0k6eGqXy+SxFfX1bDfIL5ACEY+3jFe7+9F5eAAgwEG/Er5dI4ZyzV/oyb13TQ0K6jCvAoOhBL5ROq6aBZ+s3hi

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 3: [{'type': 'text', 'text': 'For a budget-conscious client, you should recommend **Laptop A**.\n\n### Why Laptop A is the better choice:\n1. **Significant Cost Savings:** At **$650**, Laptop A saves the client **$300** (a ~31% savings) compared to Laptop B ($950).\n2. **Sufficient for Standard Tasks:** With **8 GB of RAM** and a **256 GB SSD**, it provides solid performance for everyday office tasks, web browsing, video conferencing, and general productivity.\n3. **Portability:** The smaller **14-inch display** makes it slightly more portable for travel or commuting.\n\n---\n\n### When to mention Laptop B instead:\nOnly suggest **Laptop B** if the client specifically requires power for heavy multitasking, video/photo editing, or large files that require 16 GB RAM and 512 GB SSD space. Otherwise, **Laptop A** offers the best value for money.', 'extras': {'signature': 'Ep4FCpsFARFNMg9B9BQg108JsSIjsFWZf4EMy1GDJEZCsvZilKIDskc7AE1F1d/DHruqNbagffkWz5QCov8LotQa2LP9FZPTn1ciWBfia02qak3HLJ

Turn 3 works without re-stating the laptop names because `InMemorySaver`
automatically reloads the stored history for `thread_id="budget-laptop-chat"`
before every call.

## Task 5: Structured Output & Error Handling

In [7]:
class LaptopRecommendation(BaseModel):
    recommended_product: str = Field(description="Name of the recommended laptop")
    price_usd: int = Field(description="Price of the recommended laptop in USD")
    reason: str = Field(description="One-sentence reason for the recommendation")

structured_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a helpful assistant with access to tools. "
                  "Use tools when needed, and give a clear final answer.",
    response_format=LaptopRecommendation,
)

structured_result = structured_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Between Laptop A and Laptop B, which should a budget-conscious client buy? "
        "Look up both prices first."}]}
)
print(structured_result["structured_response"])

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


recommended_product='Laptop A' price_usd=650 reason='Laptop A is significantly cheaper than Laptop B, making it the better choice for a budget-conscious client.'


In [8]:
# Error handling: get_product_price raises ToolException for unknown products.
# In LangChain v1, catch this with middleware via @wrap_tool_call.
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Catch tool exceptions and turn them into a normal ToolMessage
    the model can read, instead of letting them crash the run."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(content=f"Tool error: {e}", tool_call_id=request.tool_call["id"])

resilient_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a helpful assistant with access to tools.",
    middleware=[handle_tool_errors],
)

error_test = resilient_agent.invoke({"messages": [{"role": "user", "content": "What is the price of Laptop Z?"}]})
print("\nFINAL ANSWER:", error_test["messages"][-1].content)

d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\internship\weak2\day2\.venv3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL ANSWER: [{'type': 'text', 'text': 'I\'m sorry, but "Laptop Z" could not be found in our product database.', 'extras': {'signature': 'EpkCCpYCARFNMg+Q3pAvJ5/P/C/xsDjpbX8+TXFV38iPgRf0jzPOfVhIz9e1jiqyLCtCp8z9m3hoLK8U9sbRg4moxMyQjUXaJ03+zrorGKgZqVQUrIn47iNcNig1CeaQ0OzULQPu9KXKAY3CNz4JEjXd/dbumCTB18rMYXcYTh+0Jp/qMU3vDuYoCCt+H/4FygV7nNBQg7sbSUCthbltbGZ0EnbARZ7XKC9U5UFFXGLjf0OEoYj3m3cGLYkvpRmdHUvP7+pB3UO70lWDLwli13xqS/6PfCWRAQVAhn2wQ5HfoWZKQDx7kM0ivggPk2effIv6hI4EKHw5wgenq04xUKDKxVLh/zA2wjxav855nCgPfJ2rLGhs2ukc1g0='}}]


**How it recovers:** the `handle_tool_errors` middleware wraps every tool
call in try/except — if `get_product_price` raises, the exception is turned
into a `ToolMessage` and fed back as a normal Observation instead of
crashing `agent.invoke()`. The only config needed was
`middleware=[handle_tool_errors]`.

**LangChain vs. Day 1:** `create_agent`, `@tool`, `response_format`, and
`checkpointer` replace the hand-written prompt format, regex parser,
manual loop, and scratchpad dict. What's hidden is the exact prompt built
for tool-calling and the internal LangGraph state machine underneath.